# Feature Engineering - ReviewInsight

This notebook creates features for sentiment classification:
- TF-IDF vectorization (unigrams + bigrams)
- Numeric features (review length, temporal features)
- Binary sentiment labels


In [ ]:
import sys
import os
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'src'))

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from pathlib import Path
import pickle
import warnings
warnings.filterwarnings('ignore')

from preprocessing import preprocess_data
from modeling import create_binary_labels

# Set random seed
np.random.seed(42)


## Step 1: Load Processed Data


In [ ]:
# Load processed data
data_path = Path("../data/processed/amazon_reviews_processed.parquet")

if data_path.exists():
    print("Loading processed data...")
    df = pd.read_parquet(data_path)
else:
    print("Error: Processed data not found. Please run 01_eda.ipynb first.")
    raise FileNotFoundError("Processed data not found")

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
df.head()


## Step 2: Create Binary Sentiment Labels


In [ ]:
# Create binary labels
df_labeled = create_binary_labels(df)

print(f"\nDataset after label creation: {df_labeled.shape}")
print(f"\nLabel distribution:")
print(df_labeled['sentiment'].value_counts())
print(f"\nPercentage:")
print(df_labeled['sentiment'].value_counts(normalize=True) * 100)


## Step 3: TF-IDF Vectorization


In [ ]:
# TF-IDF vectorization with unigrams and bigrams
# Vocabulary capped at 20k features
print("Creating TF-IDF features...")
print("  - Unigrams + Bigrams")
print("  - Max features: 20,000")
print("  - Min document frequency: 2")

vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),  # Unigrams and bigrams
    min_df=2,  # Minimum document frequency
    max_df=0.95,  # Maximum document frequency (remove very common words)
    lowercase=True,
    stop_words='english'
)

# Fit and transform
X_tfidf = vectorizer.fit_transform(df_labeled['review_text_clean'])

print(f"\nTF-IDF matrix shape: {X_tfidf.shape}")
print(f"Vocabulary size: {len(vectorizer.vocabulary_)}")
print(f"Feature names (first 20): {list(vectorizer.get_feature_names_out()[:20])}")


## Step 4: Create Numeric Features


In [ ]:
# Create numeric features
numeric_features = pd.DataFrame({
    'review_length': df_labeled['review_length'],
    'review_year': df_labeled['review_year'].fillna(df_labeled['review_year'].median()),
    'review_month': df_labeled['review_month'].fillna(df_labeled['review_month'].median())
})

print("Numeric features:")
print(numeric_features.describe())

# Convert to sparse matrix for efficient concatenation
from scipy.sparse import hstack, csr_matrix

X_numeric = csr_matrix(numeric_features.values)
print(f"\nNumeric features shape: {X_numeric.shape}")


## Step 5: Combine Features


In [ ]:
# Combine TF-IDF and numeric features
X_combined = hstack([X_tfidf, X_numeric])

print(f"Combined feature matrix shape: {X_combined.shape}")
print(f"  - TF-IDF features: {X_tfidf.shape[1]}")
print(f"  - Numeric features: {X_numeric.shape[1]}")
print(f"  - Total features: {X_combined.shape[1]}")

# Create label vector
y = df_labeled['sentiment'].values

print(f"\nLabel vector shape: {y.shape}")
print(f"Label distribution: {np.bincount(y)}")


## Step 6: Save Features and Labels


In [ ]:
# Save features and labels
output_dir = Path("../outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# Save sparse matrices
from scipy.sparse import save_npz

save_npz(output_dir / "X_tfidf.npz", X_tfidf)
save_npz(output_dir / "X_combined.npz", X_combined)

# Save labels
np.save(output_dir / "y.npy", y)

# Save vectorizer
with open(output_dir / "tfidf_vectorizer.pkl", 'wb') as f:
    pickle.dump(vectorizer, f)

# Save feature names for interpretability
feature_names = list(vectorizer.get_feature_names_out()) + ['review_length', 'review_year', 'review_month']
with open(output_dir / "feature_names.pkl", 'wb') as f:
    pickle.dump(feature_names, f)

# Save metadata
metadata = {
    'n_samples': len(df_labeled),
    'n_tfidf_features': X_tfidf.shape[1],
    'n_numeric_features': X_numeric.shape[1],
    'n_total_features': X_combined.shape[1],
    'label_distribution': dict(zip(*np.unique(y, return_counts=True)))
}

with open(output_dir / "feature_metadata.pkl", 'wb') as f:
    pickle.dump(metadata, f)

print("Features and labels saved successfully!")
print(f"\nSaved files:")
print(f"  - X_tfidf.npz: TF-IDF features only")
print(f"  - X_combined.npz: Combined features (TF-IDF + numeric)")
print(f"  - y.npy: Binary labels")
print(f"  - tfidf_vectorizer.pkl: Fitted vectorizer")
print(f"  - feature_names.pkl: Feature names for interpretability")
print(f"  - feature_metadata.pkl: Metadata about features")

print(f"\nMetadata:")
for key, value in metadata.items():
    print(f"  {key}: {value}")


## Summary

Features have been successfully created:
- **TF-IDF Features**: 20,000 features from unigrams and bigrams
- **Numeric Features**: Review length, year, and month
- **Labels**: Binary sentiment (1 = positive >= 4 stars, 0 = negative <= 2 stars)

All features and labels have been saved for use in the modeling notebook.
